# ========================================
# 数据同步模块 - 基于 DataFetch
# ========================================
# 以下代码用于同步 Stock_BI 所需的各类数据表

In [2]:
# 导入所有 DataFetch 类
from DataFetch import (
    # 基础数据 (120-2000积分)
    StockBasicFetch, TradeCalFetch, StockCompanyFetch,
    StockDailyFetch, StockDailyBasicFetch,
    # 资金流向 (120-2000积分)
    MoneyFlowFetch, MoneyFlowHSGTFetch, HSGTTop10Fetch,
    # 龙虎榜 (300积分)
    TopListFetch, TopInstFetch,
    # 涨跌停 (120-5000积分)
    LimitListFetch, StkLimitFetch,
    # 指数数据 (120-2000积分)
    IndexDailyFetch, IndexBasicFetch, SWIndexDailyFetch, IndexClassifyFetch, IndexMemberFetch,
    # 港股数据 (2000积分, 注意: hk_daily需要单独购买权限1000元/年)
    HKBasicFetch, GGTDailyFetch,
    # 融资融券 (120积分)
    MarginFetch, MarginDetailFetch,
    # 财务数据 (使用VIP版本5000积分,支持period获取全市场)
    IncomeVipFetch, BalanceSheetVipFetch, CashFlowVipFetch, FinaIndicatorVipFetch,
)

from common.utils import get_engine
from sqlalchemy import text
from sqlalchemy.exc import IntegrityError
import pandas as pd
import time

engine = get_engine()

In [3]:
# 通用数据写入函数
from sqlalchemy.exc import IntegrityError  # 确保导入

def write_to_mysql(df, table_name, if_exists='append'):
    """
    将 DataFrame 写入 MySQL 表
    Args:
        df: pandas DataFrame
        table_name: 目标表名
        if_exists: 'append' 追加 / 'replace' 替换
    """
    if df is None or len(df) == 0:
        print(f"  ⚠️ 无数据跳过")
        return 0
    
    try:
        rows = df.to_sql(table_name, engine, if_exists=if_exists, index=False, method='multi')
        print(f"  ✅ 写入 {len(df)} 条记录到 {table_name}")
        return len(df)
    except IntegrityError as e:
        print(f"  ⚠️ 主键重复，部分记录已存在")
        return 0
    except Exception as e:
        print(f"  ❌ 写入失败: {e}")
        return 0


def sync_daily_data(fetcher_class, table_name, date_list, sleep_time=0.5, **kwargs):
    """
    按日期同步数据
    Args:
        fetcher_class: DataFetch 类
        table_name: 目标表名
        date_list: 日期列表 ['20250101', '20250102', ...]
        sleep_time: 请求间隔(秒)
        **kwargs: 传递给 fetch_data 的额外参数
    """
    fetcher = fetcher_class()
    total = 0
    
    for date in date_list:
        time.sleep(sleep_time)
        try:
            df = fetcher.fetch_data(f"{date}", trade_date=date, **kwargs)
            if df is not None and len(df) > 0:
                rows = write_to_mysql(df, table_name)
                total += rows
            else:
                print(f"  ⏭️ {date} 无数据")
        except Exception as e:
            print(f"  ❌ {date} 获取失败: {e}")
    
    print(f"\n📊 同步完成，共写入 {total} 条记录")
    return total

## 1. 股票基础信息（每周更新一次）

In [4]:
# 同步股票基础信息 (stock_basic)
# 建议每周更新一次
print("📦 同步股票基础信息...")
fetcher = StockBasicFetch()
df = fetcher.fetch_data("股票基础信息")
write_to_mysql(df, 'stock_basic', if_exists='replace')

📦 同步股票基础信息...
fields: ['ts_code', 'symbol', 'name', 'area', 'industry', 'market', 'exchange', 'is_hs', 'list_date']
[获取: 股票基础信息] done in 0.01 s
  ✅ 写入 5479 条记录到 stock_basic


5479

In [5]:
# 同步上市公司信息 (stock_company)
print("📦 同步上市公司信息...")
fetcher = StockCompanyFetch()
df = fetcher.fetch_data("上市公司信息")
write_to_mysql(df, 'stock_company', if_exists='replace')

📦 同步上市公司信息...
fields: ['ts_code', 'com_name', 'reg_capital', 'province', 'city', 'employees', 'main_business', 'business_scope']
[获取: 上市公司信息] done in 0.01 s
  ✅ 写入 6212 条记录到 stock_company


6212

## 2. 每日行情数据（每日更新）

In [6]:
# 设置同步日期范围
# 修改这里的日期来同步不同时间段的数据
start_date, end_date = '20260201', '20260207'

# 生成日期列表
dates = pd.date_range(start=start_date, end=end_date, freq='D')
date_list = dates.strftime('%Y%m%d').tolist()
print(f"📅 同步日期范围: {start_date} ~ {end_date}, 共 {len(date_list)} 天")

📅 同步日期范围: 20260201 ~ 20260207, 共 7 天


In [18]:
# 同步每日指标 (daily_kline)
# TuShare 接口: pro.daily()
print("📦 同步每日k线数据...")
sync_daily_data(StockDailyFetch, 'daily_kline', date_list, sleep_time=0.5)

📦 同步每日k线数据...
fields: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 20260201] done in 0.85 s
  ⏭️ 20260201 无数据
fields: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 20260202] done in 1.91 s
  ✅ 写入 5464 条记录到 daily_kline
fields: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 20260203] done in 2.55 s
  ✅ 写入 5465 条记录到 daily_kline
fields: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 20260204] done in 2.12 s
  ✅ 写入 5468 条记录到 daily_kline
fields: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 20260205] done in 2.09 s
  ✅ 写入 5467 条记录到 daily_kline
fields: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']


27330

In [7]:
# 同步每日指标 (daily_basic)
# TuShare 接口: pro.daily_basic()
print("📦 同步每日指标...")
sync_daily_data(StockDailyBasicFetch, 'daily_basic', date_list, sleep_time=0.5)

📦 同步每日指标...
fields: ['ts_code', 'trade_date', 'close', 'turnover_rate', 'turnover_rate_f', 'volume_ratio', 'pe', 'pe_ttm', 'pb', 'ps', 'ps_ttm', 'dv_ratio', 'dv_ttm', 'total_share', 'float_share', 'free_share', 'total_mv', 'circ_mv']
[获取: 20260201] done in 1.54 s
  ⏭️ 20260201 无数据
fields: ['ts_code', 'trade_date', 'close', 'turnover_rate', 'turnover_rate_f', 'volume_ratio', 'pe', 'pe_ttm', 'pb', 'ps', 'ps_ttm', 'dv_ratio', 'dv_ttm', 'total_share', 'float_share', 'free_share', 'total_mv', 'circ_mv']
[获取: 20260202] done in 2.42 s
  ✅ 写入 5464 条记录到 daily_basic
fields: ['ts_code', 'trade_date', 'close', 'turnover_rate', 'turnover_rate_f', 'volume_ratio', 'pe', 'pe_ttm', 'pb', 'ps', 'ps_ttm', 'dv_ratio', 'dv_ttm', 'total_share', 'float_share', 'free_share', 'total_mv', 'circ_mv']
[获取: 20260203] done in 2.14 s
  ✅ 写入 5465 条记录到 daily_basic
fields: ['ts_code', 'trade_date', 'close', 'turnover_rate', 'turnover_rate_f', 'volume_ratio', 'pe', 'pe_ttm', 'pb', 'ps', 'ps_ttm', 'dv_ratio', 'dv_ttm', '

27330

## 3. 资金流向数据（每日更新）

In [20]:
# 同步个股资金流向 (moneyflow)
# TuShare 接口: pro.moneyflow()
# 注意: 需要 2000+ 积分
print("📦 同步个股资金流向...")
sync_daily_data(MoneyFlowFetch, 'moneyflow', date_list, sleep_time=0.5)

📦 同步个股资金流向...
fields: ['ts_code', 'trade_date', 'buy_sm_vol', 'buy_sm_amount', 'sell_sm_vol', 'sell_sm_amount', 'buy_md_vol', 'buy_md_amount', 'sell_md_vol', 'sell_md_amount', 'buy_lg_vol', 'buy_lg_amount', 'sell_lg_vol', 'sell_lg_amount', 'buy_elg_vol', 'buy_elg_amount', 'sell_elg_vol', 'sell_elg_amount', 'net_mf_vol', 'net_mf_amount']
[获取: 20260201] done in 0.93 s
  ⏭️ 20260201 无数据
fields: ['ts_code', 'trade_date', 'buy_sm_vol', 'buy_sm_amount', 'sell_sm_vol', 'sell_sm_amount', 'buy_md_vol', 'buy_md_amount', 'sell_md_vol', 'sell_md_amount', 'buy_lg_vol', 'buy_lg_amount', 'sell_lg_vol', 'sell_lg_amount', 'buy_elg_vol', 'buy_elg_amount', 'sell_elg_vol', 'sell_elg_amount', 'net_mf_vol', 'net_mf_amount']
[获取: 20260202] done in 5.05 s
  ✅ 写入 5173 条记录到 moneyflow
fields: ['ts_code', 'trade_date', 'buy_sm_vol', 'buy_sm_amount', 'sell_sm_vol', 'sell_sm_amount', 'buy_md_vol', 'buy_md_amount', 'sell_md_vol', 'sell_md_amount', 'buy_lg_vol', 'buy_lg_amount', 'sell_lg_vol', 'sell_lg_amount', 'buy_

25876

In [21]:
# 同步沪深港通资金流向 (moneyflow_hsgt)
# TuShare 接口: pro.moneyflow_hsgt()
print("📦 同步北向资金...")
fetcher = MoneyFlowHSGTFetch()
df = fetcher.fetch_data("北向资金", start_date=start_date, end_date=end_date)
write_to_mysql(df, 'moneyflow_hsgt')

📦 同步北向资金...
fields: ['trade_date', 'ggt_ss', 'ggt_sz', 'hgt', 'sgt', 'north_money', 'south_money']
[获取: 北向资金] done in 0.84 s
  ✅ 写入 5 条记录到 moneyflow_hsgt


5

In [22]:
# 同步沪深港通十大成交股 (hsgt_top10)
# TuShare 接口: pro.hsgt_top10()
print("📦 同步沪深港通十大成交股...")
sync_daily_data(HSGTTop10Fetch, 'hsgt_top10', date_list, sleep_time=0.5)

📦 同步沪深港通十大成交股...
fields: ['trade_date', 'ts_code', 'name', 'close', 'change', 'rank', 'market_type', 'amount', 'net_amount', 'buy', 'sell']
[获取: 20260201] done in 0.84 s
  ⏭️ 20260201 无数据
fields: ['trade_date', 'ts_code', 'name', 'close', 'change', 'rank', 'market_type', 'amount', 'net_amount', 'buy', 'sell']
[获取: 20260202] done in 0.86 s
  ✅ 写入 20 条记录到 hsgt_top10
fields: ['trade_date', 'ts_code', 'name', 'close', 'change', 'rank', 'market_type', 'amount', 'net_amount', 'buy', 'sell']
[获取: 20260203] done in 1.20 s
  ✅ 写入 20 条记录到 hsgt_top10
fields: ['trade_date', 'ts_code', 'name', 'close', 'change', 'rank', 'market_type', 'amount', 'net_amount', 'buy', 'sell']
[获取: 20260204] done in 0.85 s
  ✅ 写入 20 条记录到 hsgt_top10
fields: ['trade_date', 'ts_code', 'name', 'close', 'change', 'rank', 'market_type', 'amount', 'net_amount', 'buy', 'sell']
[获取: 20260205] done in 0.88 s
  ✅ 写入 20 条记录到 hsgt_top10
fields: ['trade_date', 'ts_code', 'name', 'close', 'change', 'rank', 'market_type', 'amount', 'n

100

## 4. 龙虎榜数据（每日更新）

In [23]:
# 同步龙虎榜每日明细 (top_list)
# TuShare 接口: pro.top_list()
print("📦 同步龙虎榜...")
sync_daily_data(TopListFetch, 'top_list', date_list, sleep_time=0.5)

📦 同步龙虎榜...
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_change', 'turnover_rate', 'amount', 'l_sell', 'l_buy', 'l_amount', 'net_amount', 'net_rate', 'amount_rate', 'float_values', 'reason']
[获取: 20260201] done in 0.83 s
  ⏭️ 20260201 无数据
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_change', 'turnover_rate', 'amount', 'l_sell', 'l_buy', 'l_amount', 'net_amount', 'net_rate', 'amount_rate', 'float_values', 'reason']
[获取: 20260202] done in 0.92 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_change', 'turnover_rate', 'amount', 'l_sell', 'l_buy', 'l_amount', 'net_amount', 'net_rate', 'amount_rate', 'float_values', 'reason']
[获取: 20260203] done in 1.29 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_change', 'turnover_rate', 'amount', 'l_sell', 'l_buy', 'l_amount', 'net_amount', 'net_rate', 'amount_rate', 'float_values', 'reason']
[获取: 20260204] done in 0.97 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'na

0

In [24]:
# 同步龙虎榜机构交易明细 (top_inst)
# TuShare 接口: pro.top_inst()
print("📦 同步龙虎榜机构...")
sync_daily_data(TopInstFetch, 'top_inst', date_list, sleep_time=0.5)

📦 同步龙虎榜机构...
fields: ['trade_date', 'ts_code', 'exalter', 'buy', 'buy_rate', 'sell', 'sell_rate', 'net_buy', 'side']
[获取: 20260201] done in 0.85 s
  ⏭️ 20260201 无数据
fields: ['trade_date', 'ts_code', 'exalter', 'buy', 'buy_rate', 'sell', 'sell_rate', 'net_buy', 'side']
[获取: 20260202] done in 1.13 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'exalter', 'buy', 'buy_rate', 'sell', 'sell_rate', 'net_buy', 'side']
[获取: 20260203] done in 1.34 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'exalter', 'buy', 'buy_rate', 'sell', 'sell_rate', 'net_buy', 'side']
[获取: 20260204] done in 2.16 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'exalter', 'buy', 'buy_rate', 'sell', 'sell_rate', 'net_buy', 'side']
[获取: 20260205] done in 2.13 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'exalter', 'buy', 'buy_rate', 'sell', 'sell_rate', 'net_buy', 'side']
[获取: 20260206] done in 1.11 s
  ⚠️ 主键重复，部分记录已存在
fields: ['trade_date', 'ts_code', 'exalter', 'buy', 'buy_rate', 'sell', 'se

0

## 5. 涨跌停数据（每日更新）

In [25]:
# 同步涨跌停榜单 (limit_list)
# TuShare 接口: pro.limit_list_d()
# 注意: 需要 2000+ 积分
print("📦 同步涨跌停榜单...")
sync_daily_data(LimitListFetch, 'limit_list', date_list, sleep_time=0.5)

📦 同步涨跌停榜单...
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_chg', 'amp', 'fc_ratio', 'fl_ratio', 'fd_amount', 'first_time', 'last_time', 'open_times', 'strth', 'limit']
[获取: 20260201] done in 1.35 s
  ⏭️ 20260201 无数据
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_chg', 'amp', 'fc_ratio', 'fl_ratio', 'fd_amount', 'first_time', 'last_time', 'open_times', 'strth', 'limit']
[获取: 20260202] done in 1.03 s
  ✅ 写入 152 条记录到 limit_list
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_chg', 'amp', 'fc_ratio', 'fl_ratio', 'fd_amount', 'first_time', 'last_time', 'open_times', 'strth', 'limit']
[获取: 20260203] done in 2.31 s
  ✅ 写入 96 条记录到 limit_list
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_chg', 'amp', 'fc_ratio', 'fl_ratio', 'fd_amount', 'first_time', 'last_time', 'open_times', 'strth', 'limit']
[获取: 20260204] done in 0.99 s
  ✅ 写入 92 条记录到 limit_list
fields: ['trade_date', 'ts_code', 'name', 'close', 'pct_chg', 'amp', 'fc_ratio', 'fl_ratio', 'fd_amount', 'fir

494

In [26]:
# 同步涨跌停价格 (stk_limit)
# TuShare 接口: pro.stk_limit()
print("📦 同步涨跌停价格...")
sync_daily_data(StkLimitFetch, 'stk_limit', date_list, sleep_time=0.5)

📦 同步涨跌停价格...
fields: ['trade_date', 'ts_code', 'pre_close', 'up_limit', 'down_limit']
[获取: 20260201] done in 1.87 s
  ⏭️ 20260201 无数据
fields: ['trade_date', 'ts_code', 'pre_close', 'up_limit', 'down_limit']
[获取: 20260202] done in 3.72 s
  ✅ 写入 7455 条记录到 stk_limit
fields: ['trade_date', 'ts_code', 'pre_close', 'up_limit', 'down_limit']
[获取: 20260203] done in 2.42 s
  ✅ 写入 7458 条记录到 stk_limit
fields: ['trade_date', 'ts_code', 'pre_close', 'up_limit', 'down_limit']
[获取: 20260204] done in 1.76 s
  ✅ 写入 7458 条记录到 stk_limit
fields: ['trade_date', 'ts_code', 'pre_close', 'up_limit', 'down_limit']
[获取: 20260205] done in 2.83 s
  ✅ 写入 7459 条记录到 stk_limit
fields: ['trade_date', 'ts_code', 'pre_close', 'up_limit', 'down_limit']
[获取: 20260206] done in 1.73 s
  ✅ 写入 7464 条记录到 stk_limit
fields: ['trade_date', 'ts_code', 'pre_close', 'up_limit', 'down_limit']
[获取: 20260207] done in 1.06 s
  ⏭️ 20260207 无数据

📊 同步完成，共写入 37294 条记录


37294

## 6. 指数数据（每日更新）

In [27]:
# 同步指数基本信息 (index_basic)
# TuShare 接口: pro.index_basic()
print("📦 同步指数基本信息...")
fetcher = IndexBasicFetch()
# 分市场获取
for market in ['SSE', 'SZSE', 'SW', 'MSCI', 'CSI']:
    df = fetcher.fetch_data(f"指数-{market}", market=market)
    if df is not None and len(df) > 0:
        write_to_mysql(df, 'index_basic', 'replace')
    time.sleep(0.3)

📦 同步指数基本信息...
fields: ['ts_code', 'name', 'fullname', 'market', 'publisher', 'index_type', 'category', 'base_date', 'base_point', 'list_date', 'weight_rule', 'desc', 'exp_date']
[获取: 指数-SSE] done in 1.47 s
  ✅ 写入 594 条记录到 index_basic
fields: ['ts_code', 'name', 'fullname', 'market', 'publisher', 'index_type', 'category', 'base_date', 'base_point', 'list_date', 'weight_rule', 'desc', 'exp_date']
[获取: 指数-SZSE] done in 1.22 s
  ✅ 写入 447 条记录到 index_basic
fields: ['ts_code', 'name', 'fullname', 'market', 'publisher', 'index_type', 'category', 'base_date', 'base_point', 'list_date', 'weight_rule', 'desc', 'exp_date']
[获取: 指数-SW] done in 1.36 s
  ✅ 写入 796 条记录到 index_basic
fields: ['ts_code', 'name', 'fullname', 'market', 'publisher', 'index_type', 'category', 'base_date', 'base_point', 'list_date', 'weight_rule', 'desc', 'exp_date']
[获取: 指数-MSCI] done in 1.57 s
  ✅ 写入 3298 条记录到 index_basic
fields: ['ts_code', 'name', 'fullname', 'market', 'publisher', 'index_type', 'category', 'base_date', 'b

In [28]:
# 同步主要指数日线 (index_daily)
# TuShare 接口: pro.index_daily()
# 常用指数代码
MAIN_INDEX_CODES = [
    "000001.SH",  # 上证指数
    "399001.SZ",  # 深证成指
    "399006.SZ",  # 创业板指
    "000688.SH",  # 科创50
    "000300.SH",  # 沪深300
    "000905.SH",  # 中证500
    "000016.SH",  # 上证50
    "399005.SZ",  # 中小100
]

print("📦 同步主要指数日线...")
fetcher = IndexDailyFetch()
for ts_code in MAIN_INDEX_CODES:
    df = fetcher.fetch_data(f"指数{ts_code}", ts_code=ts_code, start_date=start_date, end_date=end_date)
    if df is not None and len(df) > 0:
        write_to_mysql(df, 'index_daily')
    time.sleep(0.3)

📦 同步主要指数日线...
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 指数000001.SH] done in 0.82 s
  ✅ 写入 5 条记录到 index_daily
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 指数399001.SZ] done in 0.82 s
  ✅ 写入 5 条记录到 index_daily
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 指数399006.SZ] done in 1.16 s
  ✅ 写入 5 条记录到 index_daily
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 指数000688.SH] done in 0.85 s
  ✅ 写入 5 条记录到 index_daily
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
[获取: 指数000300.SH] done in 0.85 s
  ✅ 写入 5 条记录到 index_daily
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'pre_close', 'change', 'pct_chg', 'vol',

In [29]:
# 同步申万行业分类 (index_classify)
# TuShare 接口: pro.index_classify()
print("📦 同步申万行业分类...")
fetcher = IndexClassifyFetch()
for level in ['L1', 'L2', 'L3']:
    df = fetcher.fetch_data(f"申万{level}", level=level, src='SW')
    if df is not None and len(df) > 0:
        write_to_mysql(df, 'index_classify')
    time.sleep(0.3)

📦 同步申万行业分类...
fields: ['index_code', 'industry_name', 'level', 'industry_code', 'is_pub', 'parent_code']
[获取: 申万L1] done in 0.84 s
fields: ['index_code', 'industry_name', 'level', 'industry_code', 'is_pub', 'parent_code']
[获取: 申万L2] done in 0.84 s
fields: ['index_code', 'industry_name', 'level', 'industry_code', 'is_pub', 'parent_code']
[获取: 申万L3] done in 0.81 s


In [30]:
# 同步申万行业指数日线 (index_sw_daily)
# TuShare 接口: pro.sw_daily()
# 注意: 需要 2000+ 积分
print("📦 同步申万行业指数日线...")
sync_daily_data(SWIndexDailyFetch, 'index_sw_daily', date_list, sleep_time=0.5)

📦 同步申万行业指数日线...
fields: ['ts_code', 'trade_date', 'name', 'open', 'low', 'high', 'close', 'change', 'pct_change', 'vol', 'amount', 'pe', 'pb']
[获取: 20260201] done in 0.86 s
  ⏭️ 20260201 无数据
fields: ['ts_code', 'trade_date', 'name', 'open', 'low', 'high', 'close', 'change', 'pct_change', 'vol', 'amount', 'pe', 'pb']
[获取: 20260202] done in 1.16 s
  ✅ 写入 439 条记录到 index_sw_daily
fields: ['ts_code', 'trade_date', 'name', 'open', 'low', 'high', 'close', 'change', 'pct_change', 'vol', 'amount', 'pe', 'pb']
[获取: 20260203] done in 2.21 s
  ✅ 写入 439 条记录到 index_sw_daily
fields: ['ts_code', 'trade_date', 'name', 'open', 'low', 'high', 'close', 'change', 'pct_change', 'vol', 'amount', 'pe', 'pb']
[获取: 20260204] done in 1.19 s
  ✅ 写入 439 条记录到 index_sw_daily
fields: ['ts_code', 'trade_date', 'name', 'open', 'low', 'high', 'close', 'change', 'pct_change', 'vol', 'amount', 'pe', 'pb']
[获取: 20260205] done in 1.20 s
  ✅ 写入 439 条记录到 index_sw_daily
fields: ['ts_code', 'trade_date', 'name', 'open', 'low', 

2195

In [31]:
# 同步申万行业成分股 (index_member)
# TuShare 接口: pro.index_member()
print("📦 同步申万行业成分股...")

# 先获取所有L1行业代码
fetcher_classify = IndexClassifyFetch()
df_classify = fetcher_classify.fetch_data("申万L1", level='L1', src='SW')

if df_classify is not None and len(df_classify) > 0:
    fetcher = IndexMemberFetch()
    for idx, row in df_classify.iterrows():
        index_code = row['index_code']
        industry_name = row.get('industry_name', '')
        print(f"  获取 {industry_name} ({index_code}) 成分股...")
        df = fetcher.fetch_data(f"成分股-{index_code}", index_code=index_code, is_new='Y')
        if df is not None and len(df) > 0:
            write_to_mysql(df, 'index_member')
        time.sleep(0.3)
else:
    print("  ⚠️ 无法获取行业分类，跳过成分股同步")

📦 同步申万行业成分股...
fields: ['index_code', 'industry_name', 'level', 'industry_code', 'is_pub', 'parent_code']
[获取: 申万L1] done in 0.00 s
  ⚠️ 无法获取行业分类，跳过成分股同步


## 7. 港股数据（每日更新）

In [32]:
# 同步港股基础信息 (hk_basic)
# TuShare 接口: pro.hk_basic()
print("📦 同步港股基础信息...")
fetcher = HKBasicFetch()
df = fetcher.fetch_data("港股基础信息", list_status='L')
write_to_mysql(df, 'hk_basic', if_exists='replace')

📦 同步港股基础信息...
fields: ['ts_code', 'name', 'fullname', 'enname', 'cn_spell', 'market', 'list_status', 'list_date', 'delist_date', 'trade_unit', 'isin', 'curr_type']
[获取: 港股基础信息] done in 1.77 s
  ✅ 写入 2715 条记录到 hk_basic


2715

In [33]:
# ⚠️ 港股日线 (hk_daily) - 需要单独购买权限！
# TuShare 接口: pro.hk_daily()
# 重要说明: 此接口需要单独购买权限 (1000元/年)，不是积分制度
# 详情参考: https://tushare.pro/document/1?doc_id=290
# 如已购买权限，请在 DataFetch/FetchHK.py 中恢复 HKDailyFetch 类并取消下方注释

# HK_CODES = [
#     "00700.HK",  # 腾讯控股
#     "09988.HK",  # 阿里巴巴
#     "03690.HK",  # 美团
#     "09618.HK",  # 京东集团
#     "09999.HK",  # 网易
#     "01810.HK",  # 小米集团
#     "02318.HK",  # 中国平安
#     "00941.HK",  # 中国移动
# ]
# from DataFetch.FetchHK import HKDailyFetch
# print("📦 同步港股日线...")
# fetcher = HKDailyFetch()
# for ts_code in HK_CODES:
#     df = fetcher.fetch_data(f"港股{ts_code}", ts_code=ts_code, start_date=start_date, end_date=end_date)
#     if df is not None and len(df) > 0:
#         write_to_mysql(df, 'hk_daily')
#     time.sleep(0.3)

print("⚠️ 港股日线(hk_daily)需要单独购买权限(1000元/年)，已跳过")

In [34]:
# 同步港股通每日成交统计 (ggt_daily) - 市场汇总数据
# TuShare 接口: pro.ggt_daily()
# 注意: 这是市场整体成交统计,不是单股数据,使用start_date/end_date获取
print("📦 同步港股通每日成交统计...")
fetcher = GGTDailyFetch()
df = fetcher.fetch_data("港股通成交统计", start_date=start_date, end_date=end_date)
write_to_mysql(df, 'ggt_daily')

📦 同步港股通每日成交统计...
fields: ['trade_date', 'buy_amount', 'buy_volume', 'sell_amount', 'sell_volume']
[获取: 港股通成交统计] done in 0.82 s
  ✅ 写入 5 条记录到 ggt_daily


5

## 8. 融资融券数据（每日更新）

In [35]:
# 同步融资融券交易汇总 (margin)
# TuShare 接口: pro.margin()
print("📦 同步融资融券汇总...")
fetcher = MarginFetch()
df = fetcher.fetch_data("融资融券汇总", start_date=start_date, end_date=end_date)
write_to_mysql(df, 'margin')

📦 同步融资融券汇总...
fields: ['trade_date', 'exchange_id', 'rzye', 'rzmre', 'rzche', 'rqye', 'rqmcl', 'rzrqye', 'rqyl']
[获取: 融资融券汇总] done in 0.84 s
  ✅ 写入 13 条记录到 margin


13

In [36]:
# 同步融资融券交易明细 (margin_detail)
# TuShare 接口: pro.margin_detail()
print("📦 同步融资融券明细...")
sync_daily_data(MarginDetailFetch, 'margin_detail', date_list, sleep_time=0.5)

📦 同步融资融券明细...
fields: ['trade_date', 'ts_code', 'name', 'rzye', 'rqye', 'rzmre', 'rqyl', 'rzche', 'rqchl', 'rqmcl', 'rzrqye']
[获取: 20260201] done in 0.79 s
  ⏭️ 20260201 无数据
fields: ['trade_date', 'ts_code', 'name', 'rzye', 'rqye', 'rzmre', 'rqyl', 'rzche', 'rqchl', 'rqmcl', 'rzrqye']
[获取: 20260202] done in 4.09 s
  ✅ 写入 4315 条记录到 margin_detail
fields: ['trade_date', 'ts_code', 'name', 'rzye', 'rqye', 'rzmre', 'rqyl', 'rzche', 'rqchl', 'rqmcl', 'rzrqye']
[获取: 20260203] done in 1.96 s
  ✅ 写入 4316 条记录到 margin_detail
fields: ['trade_date', 'ts_code', 'name', 'rzye', 'rqye', 'rzmre', 'rqyl', 'rzche', 'rqchl', 'rqmcl', 'rzrqye']
[获取: 20260204] done in 1.97 s
  ✅ 写入 4316 条记录到 margin_detail
fields: ['trade_date', 'ts_code', 'name', 'rzye', 'rqye', 'rzmre', 'rqyl', 'rzche', 'rqchl', 'rqmcl', 'rzrqye']
[获取: 20260205] done in 2.07 s
  ✅ 写入 4317 条记录到 margin_detail
fields: ['trade_date', 'ts_code', 'name', 'rzye', 'rqye', 'rzmre', 'rqyl', 'rzche', 'rqchl', 'rqmcl', 'rzrqye']
[获取: 20260206] done in

19231

## 9. 财务数据（每季度更新）

In [37]:
# 财务数据同步 - 按报告期获取
# 财报公告时间: 一季报4月、半年报8月、三季报10月、年报4月
# 建议在财报季后批量更新

# 设置要同步的报告期 (YYYYMMDD格式)
# 例如: 2024年年报='20241231', 2024年三季报='20240930'
report_periods = ['20240930', '20241231']

print(f"📅 同步财务数据, 报告期: {report_periods}")

📅 同步财务数据, 报告期: ['20240930', '20241231']


In [38]:
# 同步利润表 (income)
# TuShare 接口: pro.income_vip() - 使用VIP版本支持按period获取全市场数据
# 注意: income需要ts_code必填, income_vip(5000+积分)支持period获取全市场
print("📦 同步利润表...")
fetcher = IncomeVipFetch()  # 使用VIP版本
for period in report_periods:
    print(f"  获取 {period} 利润表...")
    df = fetcher.fetch_data(f"利润表-{period}", period=period)
    write_to_mysql(df, 'income')
    time.sleep(1)

📦 同步利润表...
  获取 20240930 利润表...
fields: ['ts_code', 'ann_date', 'end_date', 'report_type', 'comp_type', 'basic_eps', 'diluted_eps', 'total_revenue', 'revenue', 'total_cogs', 'operate_profit', 'total_profit', 'n_income', 'n_income_attr_p']
[获取: 利润表-20240930] done in 2.71 s
  ⚠️ 主键重复，部分记录已存在
  获取 20241231 利润表...
fields: ['ts_code', 'ann_date', 'end_date', 'report_type', 'comp_type', 'basic_eps', 'diluted_eps', 'total_revenue', 'revenue', 'total_cogs', 'operate_profit', 'total_profit', 'n_income', 'n_income_attr_p']
[获取: 利润表-20241231] done in 3.18 s
  ⚠️ 主键重复，部分记录已存在


In [39]:
# 同步资产负债表 (balancesheet)
# TuShare 接口: pro.balancesheet_vip() - 使用VIP版本支持按period获取全市场数据
# 注意: balancesheet需要ts_code必填, balancesheet_vip(5000+积分)支持period获取全市场
print("📦 同步资产负债表...")
fetcher = BalanceSheetVipFetch()  # 使用VIP版本
for period in report_periods:
    print(f"  获取 {period} 资产负债表...")
    df = fetcher.fetch_data(f"资产负债表-{period}", period=period)
    write_to_mysql(df, 'balancesheet')
    time.sleep(1)

📦 同步资产负债表...
  获取 20240930 资产负债表...
fields: ['ts_code', 'ann_date', 'end_date', 'report_type', 'comp_type', 'total_assets', 'total_liab', 'total_hldr_eqy_exc_min_int', 'total_hldr_eqy_inc_min_int', 'total_cur_assets', 'total_nca', 'total_cur_liab', 'total_ncl', 'accounts_receiv', 'inventories', 'money_cap']
[获取: 资产负债表-20240930] done in 2.46 s
  ⚠️ 主键重复，部分记录已存在
  获取 20241231 资产负债表...
fields: ['ts_code', 'ann_date', 'end_date', 'report_type', 'comp_type', 'total_assets', 'total_liab', 'total_hldr_eqy_exc_min_int', 'total_hldr_eqy_inc_min_int', 'total_cur_assets', 'total_nca', 'total_cur_liab', 'total_ncl', 'accounts_receiv', 'inventories', 'money_cap']
[获取: 资产负债表-20241231] done in 2.47 s
  ⚠️ 主键重复，部分记录已存在


In [40]:
# 同步现金流量表 (cashflow)
# TuShare 接口: pro.cashflow_vip() - 使用VIP版本支持按period获取全市场数据
# 注意: cashflow需要ts_code必填, cashflow_vip(5000+积分)支持period获取全市场
print("📦 同步现金流量表...")
fetcher = CashFlowVipFetch()  # 使用VIP版本
for period in report_periods:
    print(f"  获取 {period} 现金流量表...")
    df = fetcher.fetch_data(f"现金流量表-{period}", period=period)
    write_to_mysql(df, 'cashflow')
    time.sleep(1)

📦 同步现金流量表...
  获取 20240930 现金流量表...
fields: ['ts_code', 'ann_date', 'end_date', 'report_type', 'comp_type', 'net_profit', 'n_cashflow_act', 'n_cashflow_inv_act', 'n_cash_flows_fnc_act', 'c_cash_equ_end_period', 'c_cash_equ_beg_period']
[获取: 现金流量表-20240930] done in 2.60 s
  ⚠️ 主键重复，部分记录已存在
  获取 20241231 现金流量表...
fields: ['ts_code', 'ann_date', 'end_date', 'report_type', 'comp_type', 'net_profit', 'n_cashflow_act', 'n_cashflow_inv_act', 'n_cash_flows_fnc_act', 'c_cash_equ_end_period', 'c_cash_equ_beg_period']
[获取: 现金流量表-20241231] done in 2.56 s
  ⚠️ 主键重复，部分记录已存在


In [41]:
# 同步财务指标 (fina_indicator)
# TuShare 接口: pro.fina_indicator_vip() - 使用VIP版本支持按period获取全市场数据
# 注意: fina_indicator需要ts_code必填, fina_indicator_vip(5000+积分)支持period获取全市场
print("📦 同步财务指标...")
fetcher = FinaIndicatorVipFetch()  # 使用VIP版本
for period in report_periods:
    print(f"  获取 {period} 财务指标...")
    df = fetcher.fetch_data(f"财务指标-{period}", period=period)
    write_to_mysql(df, 'fina_indicator')
    time.sleep(1)

📦 同步财务指标...
  获取 20240930 财务指标...
fields: ['ts_code', 'ann_date', 'end_date', 'eps', 'dt_eps', 'bps', 'roe', 'roe_dt', 'roa', 'current_ratio', 'quick_ratio', 'gross_margin', 'netprofit_margin', 'debt_to_assets', 'op_yoy', 'profit_yoy']
[获取: 财务指标-20240930] done in 2.34 s
  ⚠️ 主键重复，部分记录已存在
  获取 20241231 财务指标...
fields: ['ts_code', 'ann_date', 'end_date', 'eps', 'dt_eps', 'bps', 'roe', 'roe_dt', 'roa', 'current_ratio', 'quick_ratio', 'gross_margin', 'netprofit_margin', 'debt_to_assets', 'op_yoy', 'profit_yoy']
[获取: 财务指标-20241231] done in 2.53 s
  ⚠️ 主键重复，部分记录已存在


## 10. 高级数据（5000+积分）

In [42]:
# 导入高级数据获取类
from DataFetch import (
    HKHoldFetch, CyqPerfFetch, StkFactorFetch, StkAuctionFetch,
    BlockTradeFetch, StkHolderNumberFetch, Top10HoldersFetch,
    Top10FloatHoldersFetch, DividendFetch, ShareFloatFetch, PledgeStatFetch,
)

In [43]:
# 同步沪深股通持股明细 (hk_hold)
# TuShare 接口: pro.hk_hold()
# 需要 5000+ 积分
print("📦 同步沪深股通持股明细...")
sync_daily_data(HKHoldFetch, 'hk_hold', date_list, sleep_time=0.5)

📦 同步沪深股通持股明细...
fields: ['trade_date', 'ts_code', 'name', 'vol', 'ratio', 'exchange']
[获取: 20260201] done in 0.85 s
  ⏭️ 20260201 无数据
fields: ['trade_date', 'ts_code', 'name', 'vol', 'ratio', 'exchange']
[获取: 20260202] done in 1.15 s
  ✅ 写入 874 条记录到 hk_hold
fields: ['trade_date', 'ts_code', 'name', 'vol', 'ratio', 'exchange']
[获取: 20260203] done in 1.29 s
  ✅ 写入 874 条记录到 hk_hold
fields: ['trade_date', 'ts_code', 'name', 'vol', 'ratio', 'exchange']
[获取: 20260204] done in 1.14 s
  ✅ 写入 874 条记录到 hk_hold
fields: ['trade_date', 'ts_code', 'name', 'vol', 'ratio', 'exchange']
[获取: 20260205] done in 1.40 s
  ✅ 写入 874 条记录到 hk_hold
fields: ['trade_date', 'ts_code', 'name', 'vol', 'ratio', 'exchange']
[获取: 20260206] done in 1.12 s
  ✅ 写入 875 条记录到 hk_hold
fields: ['trade_date', 'ts_code', 'name', 'vol', 'ratio', 'exchange']
[获取: 20260207] done in 0.80 s
  ⏭️ 20260207 无数据

📊 同步完成，共写入 4371 条记录


4371

In [44]:
# 同步每日筹码及胜率 (cyq_perf)
# TuShare 接口: pro.cyq_perf()
# 需要 5000+ 积分
print("📦 同步每日筹码及胜率...")
sync_daily_data(CyqPerfFetch, 'cyq_perf', date_list, sleep_time=0.5)

📦 同步每日筹码及胜率...
fields: ['ts_code', 'trade_date', 'his_low', 'his_high', 'cost_5pct', 'cost_15pct', 'cost_50pct', 'cost_85pct', 'cost_95pct', 'weight_avg', 'winner_rate']
[获取: 20260201] done in 0.81 s
  ⏭️ 20260201 无数据
fields: ['ts_code', 'trade_date', 'his_low', 'his_high', 'cost_5pct', 'cost_15pct', 'cost_50pct', 'cost_85pct', 'cost_95pct', 'weight_avg', 'winner_rate']
[获取: 20260202] done in 2.96 s
  ✅ 写入 5464 条记录到 cyq_perf
fields: ['ts_code', 'trade_date', 'his_low', 'his_high', 'cost_5pct', 'cost_15pct', 'cost_50pct', 'cost_85pct', 'cost_95pct', 'weight_avg', 'winner_rate']
[获取: 20260203] done in 2.44 s
  ✅ 写入 5465 条记录到 cyq_perf
fields: ['ts_code', 'trade_date', 'his_low', 'his_high', 'cost_5pct', 'cost_15pct', 'cost_50pct', 'cost_85pct', 'cost_95pct', 'weight_avg', 'winner_rate']
[获取: 20260204] done in 2.08 s
  ✅ 写入 5468 条记录到 cyq_perf
fields: ['ts_code', 'trade_date', 'his_low', 'his_high', 'cost_5pct', 'cost_15pct', 'cost_50pct', 'cost_85pct', 'cost_95pct', 'weight_avg', 'winner_r

27330

In [45]:
# 同步股票技术面因子 (stk_factor)
# TuShare 接口: pro.stk_factor_pro()
# 需要 5000+ 积分
print("📦 同步股票技术面因子...")
sync_daily_data(StkFactorFetch, 'stk_factor', date_list, sleep_time=0.5)

📦 同步股票技术面因子...
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'vol', 'amount', 'macd_dif', 'macd_dea', 'macd', 'kdj_k', 'kdj_d', 'kdj_j', 'rsi_6', 'rsi_12', 'rsi_24', 'boll_upper', 'boll_mid', 'boll_lower', 'cci']
[获取: 20260201] done in 0.85 s
  ⏭️ 20260201 无数据
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'vol', 'amount', 'macd_dif', 'macd_dea', 'macd', 'kdj_k', 'kdj_d', 'kdj_j', 'rsi_6', 'rsi_12', 'rsi_24', 'boll_upper', 'boll_mid', 'boll_lower', 'cci']
[获取: 20260202] done in 1.99 s
  ✅ 写入 5464 条记录到 stk_factor
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'vol', 'amount', 'macd_dif', 'macd_dea', 'macd', 'kdj_k', 'kdj_d', 'kdj_j', 'rsi_6', 'rsi_12', 'rsi_24', 'boll_upper', 'boll_mid', 'boll_lower', 'cci']
[获取: 20260203] done in 1.78 s
  ✅ 写入 5465 条记录到 stk_factor
fields: ['ts_code', 'trade_date', 'close', 'open', 'high', 'low', 'vol', 'amount', 'macd_dif', 'macd_dea', 'macd', 'kdj_k', 'kdj_d', 'kdj_j', 'rsi_6', 'rsi_12', 'rsi_2

27330

In [46]:
# 同步大宗交易 (block_trade)
# TuShare 接口: pro.block_trade()
# 需要 2000+ 积分
print("📦 同步大宗交易...")
sync_daily_data(BlockTradeFetch, 'block_trade', date_list, sleep_time=0.5)

📦 同步大宗交易...
fields: ['ts_code', 'trade_date', 'name', 'price', 'vol', 'amount', 'buyer', 'seller']
[获取: 20260201] done in 0.80 s
  ⏭️ 20260201 无数据
fields: ['ts_code', 'trade_date', 'name', 'price', 'vol', 'amount', 'buyer', 'seller']
[获取: 20260202] done in 0.95 s
  ⚠️ 主键重复，部分记录已存在
fields: ['ts_code', 'trade_date', 'name', 'price', 'vol', 'amount', 'buyer', 'seller']
[获取: 20260203] done in 0.84 s
  ⚠️ 主键重复，部分记录已存在
fields: ['ts_code', 'trade_date', 'name', 'price', 'vol', 'amount', 'buyer', 'seller']
[获取: 20260204] done in 0.85 s
  ⚠️ 主键重复，部分记录已存在
fields: ['ts_code', 'trade_date', 'name', 'price', 'vol', 'amount', 'buyer', 'seller']
[获取: 20260205] done in 0.79 s
  ⚠️ 主键重复，部分记录已存在
fields: ['ts_code', 'trade_date', 'name', 'price', 'vol', 'amount', 'buyer', 'seller']
[获取: 20260206] done in 0.85 s
  ⚠️ 主键重复，部分记录已存在
fields: ['ts_code', 'trade_date', 'name', 'price', 'vol', 'amount', 'buyer', 'seller']
[获取: 20260207] done in 0.86 s
  ⏭️ 20260207 无数据

📊 同步完成，共写入 0 条记录


0

In [47]:
# 同步股东人数 (stk_holdernumber)
# TuShare 接口: pro.stk_holdernumber()
# 需要 2000+ 积分，按报告期获取
print("📦 同步股东人数...")
fetcher = StkHolderNumberFetch()
for period in report_periods:
    print(f"  获取 {period} 股东人数...")
    df = fetcher.fetch_data(f"股东人数-{period}", end_date=period)
    write_to_mysql(df, 'stk_holdernumber')
    time.sleep(1)

📦 同步股东人数...
  获取 20240930 股东人数...
fields: ['ts_code', 'ann_date', 'end_date', 'holder_num', 'holder_num_change', 'holder_num_ratio', 'holder_num_pct']
[获取: 股东人数-20240930] done in 1.52 s
  ⚠️ 主键重复，部分记录已存在
  获取 20241231 股东人数...
fields: ['ts_code', 'ann_date', 'end_date', 'holder_num', 'holder_num_change', 'holder_num_ratio', 'holder_num_pct']
[获取: 股东人数-20241231] done in 1.51 s
  ⚠️ 主键重复，部分记录已存在


In [48]:
# 同步前十大股东 (top10_holders)
# TuShare 接口: pro.top10_holders()
# 需要 2000+ 积分，按报告期获取
print("📦 同步前十大股东...")
fetcher = Top10HoldersFetch()
for period in report_periods:
    print(f"  获取 {period} 前十大股东...")
    df = fetcher.fetch_data(f"十大股东-{period}", period=period)
    write_to_mysql(df, 'top10_holders')
    time.sleep(1)

📦 同步前十大股东...
  获取 20240930 前十大股东...
fields: ['ts_code', 'ann_date', 'end_date', 'holder_name', 'hold_amount', 'hold_ratio', 'hold_change', 'holder_type']
[获取: 十大股东-20240930] done in 2.31 s
  ⚠️ 主键重复，部分记录已存在
  获取 20241231 前十大股东...
fields: ['ts_code', 'ann_date', 'end_date', 'holder_name', 'hold_amount', 'hold_ratio', 'hold_change', 'holder_type']
[获取: 十大股东-20241231] done in 2.17 s
  ⚠️ 主键重复，部分记录已存在


In [49]:
# 同步前十大流通股东 (top10_floatholders)
# TuShare 接口: pro.top10_floatholders()
# 需要 2000+ 积分，按报告期获取
print("📦 同步前十大流通股东...")
fetcher = Top10FloatHoldersFetch()
for period in report_periods:
    print(f"  获取 {period} 十大流通股东...")
    df = fetcher.fetch_data(f"十大流通股东-{period}", period=period)
    write_to_mysql(df, 'top10_floatholders')
    time.sleep(1)

📦 同步前十大流通股东...
  获取 20240930 十大流通股东...
fields: ['ts_code', 'ann_date', 'end_date', 'holder_name', 'hold_amount', 'hold_ratio', 'hold_change', 'holder_type']
[获取: 十大流通股东-20240930] done in 3.48 s
  ⚠️ 主键重复，部分记录已存在
  获取 20241231 十大流通股东...
fields: ['ts_code', 'ann_date', 'end_date', 'holder_name', 'hold_amount', 'hold_ratio', 'hold_change', 'holder_type']
[获取: 十大流通股东-20241231] done in 2.46 s
  ⚠️ 主键重复，部分记录已存在


In [50]:
# 同步分红送股 (dividend)
# TuShare 接口: pro.dividend()
# 需要 2000+ 积分
print("📦 同步分红送股...")
fetcher = DividendFetch()
# 获取最近几年的分红数据
for year in ['2024', '2023', '2022']:
    print(f"  获取 {year} 年分红数据...")
    df = fetcher.fetch_data(f"分红-{year}", ann_date=f"{year}0101", end_date=f"{year}1231")
    if df is None or len(df) == 0:
        # 尝试用 ex_date 参数
        pass
    write_to_mysql(df, 'dividend')
    time.sleep(1)

📦 同步分红送股...
  获取 2024 年分红数据...
fields: ['ts_code', 'ann_date', 'end_date', 'div_proc', 'stk_div', 'stk_bo_rate', 'stk_co_rate', 'cash_div', 'cash_div_tax', 'record_date', 'ex_date', 'pay_date']
[获取: 分红-2024] done in 1.45 s
  ⚠️ 无数据跳过
  获取 2023 年分红数据...
fields: ['ts_code', 'ann_date', 'end_date', 'div_proc', 'stk_div', 'stk_bo_rate', 'stk_co_rate', 'cash_div', 'cash_div_tax', 'record_date', 'ex_date', 'pay_date']
[获取: 分红-2023] done in 0.84 s
  ⚠️ 无数据跳过
  获取 2022 年分红数据...
fields: ['ts_code', 'ann_date', 'end_date', 'div_proc', 'stk_div', 'stk_bo_rate', 'stk_co_rate', 'cash_div', 'cash_div_tax', 'record_date', 'ex_date', 'pay_date']
[获取: 分红-2022] done in 0.88 s
  ⚠️ 无数据跳过


In [51]:
# 同步限售股解禁 (share_float)
# TuShare 接口: pro.share_float()
# 需要 3000+ 积分
print("📦 同步限售股解禁...")
fetcher = ShareFloatFetch()
df = fetcher.fetch_data("限售股解禁", start_date=start_date, end_date=end_date)
write_to_mysql(df, 'share_float')

📦 同步限售股解禁...
fields: ['ts_code', 'ann_date', 'float_date', 'float_share', 'float_ratio', 'holder_name', 'share_type']
[获取: 限售股解禁] done in 2.11 s
  ⚠️ 主键重复，部分记录已存在


0

In [52]:
# 同步股权质押统计 (pledge_stat)
# TuShare 接口: pro.pledge_stat()
# 需要 2000+ 积分
print("📦 同步股权质押统计...")
fetcher = PledgeStatFetch()
# 获取最新数据
df = fetcher.fetch_data("股权质押", end_date=end_date)
write_to_mysql(df, 'pledge_stat')

📦 同步股权质押统计...
fields: ['ts_code', 'end_date', 'pledge_count', 'unrest_pledge', 'rest_pledge', 'total_share', 'pledge_ratio']
[获取: 股权质押] done in 0.88 s
  ⚠️ 无数据跳过


0

## 10. 建表脚本（首次使用时运行）
执行以下SQL创建所有数据表，包含索引和注释。

In [53]:
# 建表SQL脚本 - 首次使用时执行
CREATE_TABLES_SQL = """
-- =====================================================
-- Stock BI 数据表建表脚本
-- 生成时间: 2025-01-24
-- 数据源: TuShare Pro API + DataFetch 模块
-- =====================================================

-- 1. A股日K线
CREATE TABLE IF NOT EXISTS daily_kline (
    ts_code     VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    open        FLOAT                  COMMENT '开盘价',
    high        FLOAT                  COMMENT '最高价',
    low         FLOAT                  COMMENT '最低价',
    close       FLOAT                  COMMENT '收盘价',
    pre_close   FLOAT                  COMMENT '昨收价(除权价)',
    `change`    FLOAT                  COMMENT '涨跌额',
    pct_chg     FLOAT                  COMMENT '涨跌幅(%)',
    vol         FLOAT                  COMMENT '成交量(手)',
    amount      FLOAT                  COMMENT '成交额(千元)',
    PRIMARY KEY (ts_code, trade_date),
    INDEX idx_trade_date (trade_date),
    INDEX idx_date_pct (trade_date, pct_chg),
    INDEX idx_date_amount (trade_date, amount)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='A股日K线数据';

-- 2. 每日指标
CREATE TABLE IF NOT EXISTS daily_basic (
    ts_code         VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    trade_date      VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    close           FLOAT                  COMMENT '当日收盘价',
    turnover_rate   FLOAT                  COMMENT '换手率(%)',
    turnover_rate_f FLOAT                  COMMENT '换手率(自由流通股)',
    volume_ratio    FLOAT                  COMMENT '量比',
    pe              FLOAT                  COMMENT '市盈率(总市值/净利润)',
    pe_ttm          FLOAT                  COMMENT '市盈率TTM',
    pb              FLOAT                  COMMENT '市净率',
    ps              FLOAT                  COMMENT '市销率',
    ps_ttm          FLOAT                  COMMENT '市销率TTM',
    dv_ratio        FLOAT                  COMMENT '股息率(%)',
    dv_ttm          FLOAT                  COMMENT '股息率TTM(%)',
    total_share     FLOAT                  COMMENT '总股本(万股)',
    float_share     FLOAT                  COMMENT '流通股本(万股)',
    free_share      FLOAT                  COMMENT '自由流通股本(万股)',
    total_mv        FLOAT                  COMMENT '总市值(万元)',
    circ_mv         FLOAT                  COMMENT '流通市值(万元)',
    PRIMARY KEY (ts_code, trade_date),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='每日指标数据';

-- 3. 股票基础信息
CREATE TABLE IF NOT EXISTS stock_basic (
    ts_code     VARCHAR(10)   NOT NULL  COMMENT '股票代码',
    symbol      VARCHAR(10)             COMMENT '股票代码',
    name        VARCHAR(50)             COMMENT '股票名称',
    area        VARCHAR(20)             COMMENT '地域',
    industry    VARCHAR(50)             COMMENT '所属行业',
    market      VARCHAR(10)             COMMENT '市场类别',
    exchange    VARCHAR(10)             COMMENT '交易所代码',
    is_hs       VARCHAR(5)              COMMENT '是否沪深港通',
    list_date   VARCHAR(8)              COMMENT '上市日期',
    PRIMARY KEY (ts_code),
    INDEX idx_market (market),
    INDEX idx_industry (industry)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='股票基础信息';

-- 4. 交易日历
CREATE TABLE IF NOT EXISTS trade_cal (
    exchange    VARCHAR(10)  NOT NULL  COMMENT '交易所',
    cal_date    VARCHAR(8)   NOT NULL  COMMENT '日期(YYYYMMDD)',
    is_open     VARCHAR(1)             COMMENT '是否交易(0/1)',
    PRIMARY KEY (exchange, cal_date),
    INDEX idx_cal_date (cal_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='交易日历';

-- 5. 上市公司基本信息
CREATE TABLE IF NOT EXISTS stock_company (
    ts_code         VARCHAR(10)   NOT NULL  COMMENT '股票代码',
    com_name        VARCHAR(100)            COMMENT '公司名称',
    reg_capital     FLOAT                   COMMENT '注册资本(万元)',
    province        VARCHAR(20)             COMMENT '省份',
    city            VARCHAR(30)             COMMENT '城市',
    employees       INT                     COMMENT '员工人数',
    main_business   TEXT                    COMMENT '主要业务',
    business_scope  TEXT                    COMMENT '经营范围',
    PRIMARY KEY (ts_code)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='上市公司基本信息';

-- 6. 个股资金流向
CREATE TABLE IF NOT EXISTS moneyflow (
    ts_code         VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    trade_date      VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    buy_sm_vol      FLOAT                  COMMENT '小单买入量(手)',
    buy_sm_amount   FLOAT                  COMMENT '小单买入金额(万元)',
    sell_sm_vol     FLOAT                  COMMENT '小单卖出量(手)',
    sell_sm_amount  FLOAT                  COMMENT '小单卖出金额(万元)',
    buy_md_vol      FLOAT                  COMMENT '中单买入量(手)',
    buy_md_amount   FLOAT                  COMMENT '中单买入金额(万元)',
    sell_md_vol     FLOAT                  COMMENT '中单卖出量(手)',
    sell_md_amount  FLOAT                  COMMENT '中单卖出金额(万元)',
    buy_lg_vol      FLOAT                  COMMENT '大单买入量(手)',
    buy_lg_amount   FLOAT                  COMMENT '大单买入金额(万元)',
    sell_lg_vol     FLOAT                  COMMENT '大单卖出量(手)',
    sell_lg_amount  FLOAT                  COMMENT '大单卖出金额(万元)',
    buy_elg_vol     FLOAT                  COMMENT '特大单买入量(手)',
    buy_elg_amount  FLOAT                  COMMENT '特大单买入金额(万元)',
    sell_elg_vol    FLOAT                  COMMENT '特大单卖出量(手)',
    sell_elg_amount FLOAT                  COMMENT '特大单卖出金额(万元)',
    net_mf_vol      FLOAT                  COMMENT '净流入量(手)',
    net_mf_amount   FLOAT                  COMMENT '净流入额(万元)',
    PRIMARY KEY (ts_code, trade_date),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='个股资金流向';

-- 7. 沪深港通资金流向
CREATE TABLE IF NOT EXISTS moneyflow_hsgt (
    trade_date   VARCHAR(8)  NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ggt_ss       FLOAT                 COMMENT '港股通(沪)净流入(百万)',
    ggt_sz       FLOAT                 COMMENT '港股通(深)净流入(百万)',
    hgt          FLOAT                 COMMENT '沪股通净流入(百万)',
    sgt          FLOAT                 COMMENT '深股通净流入(百万)',
    north_money  FLOAT                 COMMENT '北向资金净流入(百万)',
    south_money  FLOAT                 COMMENT '南向资金净流入(百万)',
    PRIMARY KEY (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='沪深港通资金流向';

-- 8. 沪深港通十大成交股
CREATE TABLE IF NOT EXISTS hsgt_top10 (
    trade_date   VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ts_code      VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    name         VARCHAR(50)            COMMENT '股票名称',
    close        FLOAT                  COMMENT '收盘价',
    `change`     FLOAT                  COMMENT '涨跌额',
    `rank`       INT                    COMMENT '资金排名',
    market_type  VARCHAR(5)   NOT NULL  COMMENT '市场类型(1沪/2深/3港沪/4港深)',
    amount       FLOAT                  COMMENT '成交金额(百万)',
    net_amount   FLOAT                  COMMENT '净买入金额(百万)',
    buy          FLOAT                  COMMENT '买入金额(百万)',
    sell         FLOAT                  COMMENT '卖出金额(百万)',
    PRIMARY KEY (trade_date, ts_code, market_type),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='沪深港通十大成交股';

-- 9. 龙虎榜每日明细
CREATE TABLE IF NOT EXISTS top_list (
    trade_date    VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ts_code       VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    name          VARCHAR(50)            COMMENT '股票名称',
    close         FLOAT                  COMMENT '收盘价',
    pct_change    FLOAT                  COMMENT '涨跌幅(%)',
    turnover_rate FLOAT                  COMMENT '换手率(%)',
    amount        FLOAT                  COMMENT '总成交额(万)',
    l_sell        FLOAT                  COMMENT '龙虎榜卖出额(万)',
    l_buy         FLOAT                  COMMENT '龙虎榜买入额(万)',
    l_amount      FLOAT                  COMMENT '龙虎榜成交额(万)',
    net_amount    FLOAT                  COMMENT '龙虎榜净买入额(万)',
    net_rate      FLOAT                  COMMENT '净买入占比(%)',
    amount_rate   FLOAT                  COMMENT '龙虎榜成交额占比(%)',
    float_values  FLOAT                  COMMENT '当日流通市值(万)',
    reason        VARCHAR(200)           COMMENT '上榜原因',
    PRIMARY KEY (trade_date, ts_code),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='龙虎榜每日明细';

-- 10. 龙虎榜机构交易明细
CREATE TABLE IF NOT EXISTS top_inst (
    trade_date  VARCHAR(8)    NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ts_code     VARCHAR(10)   NOT NULL  COMMENT '股票代码',
    exalter     VARCHAR(200)  NOT NULL  COMMENT '营业部名称',
    buy         FLOAT                   COMMENT '买入额(万)',
    buy_rate    FLOAT                   COMMENT '买入占总成交比例(%)',
    sell        FLOAT                   COMMENT '卖出额(万)',
    sell_rate   FLOAT                   COMMENT '卖出占总成交比例(%)',
    net_buy     FLOAT                   COMMENT '净买入额(万)',
    side        VARCHAR(10)             COMMENT '买卖方向(BUY/SELL)',
    PRIMARY KEY (trade_date, ts_code, exalter(100)),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='龙虎榜机构交易明细';

-- 11. 涨跌停榜单
CREATE TABLE IF NOT EXISTS limit_list (
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ts_code     VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    name        VARCHAR(50)            COMMENT '股票名称',
    close       FLOAT                  COMMENT '收盘价',
    pct_chg     FLOAT                  COMMENT '涨跌幅(%)',
    amp         FLOAT                  COMMENT '振幅(%)',
    fc_ratio    FLOAT                  COMMENT '封单金额/流通市值',
    fl_ratio    FLOAT                  COMMENT '封单手数/流通股本',
    fd_amount   FLOAT                  COMMENT '封单金额(万)',
    first_time  VARCHAR(10)            COMMENT '首次涨跌停时间',
    last_time   VARCHAR(10)            COMMENT '最后涨跌停时间',
    open_times  INT                    COMMENT '打开次数',
    strth       FLOAT                  COMMENT '涨跌停强度',
    `limit`     VARCHAR(5)             COMMENT '涨跌停类型(U/D/Z)',
    PRIMARY KEY (trade_date, ts_code),
    INDEX idx_trade_date (trade_date),
    INDEX idx_date_limit (trade_date, `limit`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='涨跌停榜单';

-- 12. 每日涨跌停价格
CREATE TABLE IF NOT EXISTS stk_limit (
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ts_code     VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    pre_close   FLOAT                  COMMENT '昨日收盘价',
    up_limit    FLOAT                  COMMENT '涨停价',
    down_limit  FLOAT                  COMMENT '跌停价',
    PRIMARY KEY (trade_date, ts_code),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='每日涨跌停价格';

-- 13. 指数日线行情
CREATE TABLE IF NOT EXISTS index_daily (
    ts_code     VARCHAR(15)  NOT NULL  COMMENT '指数代码',
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    close       FLOAT                  COMMENT '收盘点位',
    open        FLOAT                  COMMENT '开盘点位',
    high        FLOAT                  COMMENT '最高点位',
    low         FLOAT                  COMMENT '最低点位',
    pre_close   FLOAT                  COMMENT '昨收点位',
    `change`    FLOAT                  COMMENT '涨跌点',
    pct_chg     FLOAT                  COMMENT '涨跌幅(%)',
    vol         FLOAT                  COMMENT '成交量(手)',
    amount      FLOAT                  COMMENT '成交额(千元)',
    PRIMARY KEY (ts_code, trade_date),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='指数日线行情';

-- 14. 指数基本信息
CREATE TABLE IF NOT EXISTS index_basic (
    ts_code     VARCHAR(30)   NOT NULL  COMMENT '指数代码',
    name        VARCHAR(100)            COMMENT '简称',
    fullname    VARCHAR(200)            COMMENT '指数全称',
    market      VARCHAR(20)             COMMENT '市场',
    publisher   VARCHAR(50)             COMMENT '发布方',
    index_type  VARCHAR(20)             COMMENT '指数类型',
    category    VARCHAR(20)             COMMENT '指数类别',
    base_date   VARCHAR(8)              COMMENT '基期',
    base_point  FLOAT                   COMMENT '基点',
    list_date   VARCHAR(8)              COMMENT '发布日期',
    weight_rule VARCHAR(50)             COMMENT '加权方式',
    `desc`      TEXT                    COMMENT '描述',
    exp_date    VARCHAR(8)              COMMENT '终止日期',
    PRIMARY KEY (ts_code)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='指数基本信息';

-- 15. 申万行业指数日线
CREATE TABLE IF NOT EXISTS index_sw_daily (
    ts_code     VARCHAR(15)  NOT NULL  COMMENT '申万行业指数代码',
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    name        VARCHAR(50)            COMMENT '行业名称',
    open        FLOAT                  COMMENT '开盘点位',
    low         FLOAT                  COMMENT '最低点位',
    high        FLOAT                  COMMENT '最高点位',
    close       FLOAT                  COMMENT '收盘点位',
    `change`    FLOAT                  COMMENT '涨跌点',
    pct_change  FLOAT                  COMMENT '涨跌幅(%)',
    vol         FLOAT                  COMMENT '成交量(手)',
    amount      FLOAT                  COMMENT '成交额(千元)',
    pe          FLOAT                  COMMENT '市盈率',
    pb          FLOAT                  COMMENT '市净率',
    PRIMARY KEY (ts_code, trade_date),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='申万行业指数日线';

-- 16. 申万行业分类
CREATE TABLE IF NOT EXISTS index_classify (
    index_code     VARCHAR(30)  NOT NULL  COMMENT '指数代码',
    industry_name  VARCHAR(50)            COMMENT '行业名称',
    level          VARCHAR(5)             COMMENT '行业级别(L1/L2/L3)',
    industry_code  VARCHAR(20)            COMMENT '行业代码',
    is_pub         VARCHAR(5)             COMMENT '是否发布指数',
    parent_code    VARCHAR(20)            COMMENT '父级代码',
    PRIMARY KEY (index_code)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='申万行业分类';

-- 17. 申万行业成分股
CREATE TABLE IF NOT EXISTS index_member (
    index_code  VARCHAR(30)  NOT NULL  COMMENT '指数代码',
    index_name  VARCHAR(50)            COMMENT '指数名称',
    con_code    VARCHAR(10)  NOT NULL  COMMENT '成分股代码',
    con_name    VARCHAR(50)            COMMENT '成分股名称',
    in_date     VARCHAR(8)             COMMENT '纳入日期',
    out_date    VARCHAR(8)             COMMENT '剔除日期',
    is_new      VARCHAR(5)             COMMENT '是否最新(Y/N)',
    PRIMARY KEY (index_code, con_code),
    INDEX idx_con_code (con_code)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='申万行业成分股';

-- 18. 港股日线行情 (需要单独购买权限1000元/年，此处保留DDL供参考)
-- CREATE TABLE IF NOT EXISTS hk_daily (
--     ts_code     VARCHAR(15)  NOT NULL  COMMENT '港股代码',
--     trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
--     open        FLOAT                  COMMENT '开盘价',
--     high        FLOAT                  COMMENT '最高价',
--     low         FLOAT                  COMMENT '最低价',
--     close       FLOAT                  COMMENT '收盘价',
--     pre_close   FLOAT                  COMMENT '昨收价',
--     `change`    FLOAT                  COMMENT '涨跌额',
--     pct_chg     FLOAT                  COMMENT '涨跌幅(%)',
--     vol         FLOAT                  COMMENT '成交量(手)',
--     amount      FLOAT                  COMMENT '成交额(千港元)',
--     PRIMARY KEY (ts_code, trade_date),
--     INDEX idx_trade_date (trade_date)
-- ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='港股日线行情';

-- 19. 港股基础信息
CREATE TABLE IF NOT EXISTS hk_basic (
    ts_code      VARCHAR(15)   NOT NULL  COMMENT 'TS代码',
    name         VARCHAR(100)            COMMENT '股票名称',
    fullname     VARCHAR(200)            COMMENT '公司全称',
    enname       VARCHAR(200)            COMMENT '英文名称',
    cn_spell     VARCHAR(50)             COMMENT '拼音',
    market       VARCHAR(10)             COMMENT '市场类别',
    list_status  VARCHAR(5)              COMMENT '上市状态(L/D/P)',
    list_date    VARCHAR(8)              COMMENT '上市日期',
    delist_date  VARCHAR(8)              COMMENT '退市日期',
    trade_unit   INT                     COMMENT '交易单位(股)',
    isin         VARCHAR(20)             COMMENT 'ISIN代码',
    curr_type    VARCHAR(10)             COMMENT '交易货币',
    PRIMARY KEY (ts_code)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='港股基础信息';

-- 20. 港股通每日成交统计(市场汇总)
CREATE TABLE IF NOT EXISTS ggt_daily (
    trade_date   VARCHAR(8)  NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    buy_amount   FLOAT                 COMMENT '买入成交金额(亿元)',
    buy_volume   FLOAT                 COMMENT '买入成交笔数(万笔)',
    sell_amount  FLOAT                 COMMENT '卖出成交金额(亿元)',
    sell_volume  FLOAT                 COMMENT '卖出成交笔数(万笔)',
    PRIMARY KEY (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='港股通每日成交统计(市场汇总)';

-- 21. 融资融券交易汇总
CREATE TABLE IF NOT EXISTS margin (
    trade_date   VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    exchange_id  VARCHAR(10)  NOT NULL  COMMENT '交易所(SSE/SZSE)',
    rzye         FLOAT                  COMMENT '融资余额(元)',
    rzmre        FLOAT                  COMMENT '融资买入额(元)',
    rzche        FLOAT                  COMMENT '融资偿还额(元)',
    rqye         FLOAT                  COMMENT '融券余额(元)',
    rqmcl        FLOAT                  COMMENT '融券卖出量(股)',
    rzrqye       FLOAT                  COMMENT '融资融券余额(元)',
    rqyl         FLOAT                  COMMENT '融券余量(股)',
    PRIMARY KEY (trade_date, exchange_id),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='融资融券交易汇总';

-- 22. 融资融券交易明细
CREATE TABLE IF NOT EXISTS margin_detail (
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ts_code     VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    name        VARCHAR(50)            COMMENT '股票名称',
    rzye        FLOAT                  COMMENT '融资余额(元)',
    rqye        FLOAT                  COMMENT '融券余额(元)',
    rzmre       FLOAT                  COMMENT '融资买入额(元)',
    rqyl        FLOAT                  COMMENT '融券余量(股)',
    rzche       FLOAT                  COMMENT '融资偿还额(元)',
    rqchl       FLOAT                  COMMENT '融券偿还量(股)',
    rqmcl       FLOAT                  COMMENT '融券卖出量(股)',
    rzrqye      FLOAT                  COMMENT '融资融券余额(元)',
    PRIMARY KEY (trade_date, ts_code),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='融资融券交易明细';

-- 23. 利润表
CREATE TABLE IF NOT EXISTS income (
    ts_code         VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    ann_date        VARCHAR(8)             COMMENT '公告日期',
    end_date        VARCHAR(8)   NOT NULL  COMMENT '报告期(YYYYMMDD)',
    report_type     VARCHAR(5)   NOT NULL  COMMENT '报告类型',
    comp_type       VARCHAR(5)             COMMENT '公司类型',
    basic_eps       FLOAT                  COMMENT '基本每股收益(元)',
    diluted_eps     FLOAT                  COMMENT '稀释每股收益(元)',
    total_revenue   FLOAT                  COMMENT '营业总收入(元)',
    revenue         FLOAT                  COMMENT '营业收入(元)',
    total_cogs      FLOAT                  COMMENT '营业总成本(元)',
    operate_profit  FLOAT                  COMMENT '营业利润(元)',
    total_profit    FLOAT                  COMMENT '利润总额(元)',
    n_income        FLOAT                  COMMENT '净利润(元)',
    n_income_attr_p FLOAT                  COMMENT '归母净利润(元)',
    ebit            FLOAT                  COMMENT '息税前利润(元)',
    ebitda          FLOAT                  COMMENT '息税折旧摊销前利润(元)',
    PRIMARY KEY (ts_code, end_date, report_type),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='利润表';

-- 24. 资产负债表
CREATE TABLE IF NOT EXISTS balancesheet (
    ts_code                     VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    ann_date                    VARCHAR(8)             COMMENT '公告日期',
    end_date                    VARCHAR(8)   NOT NULL  COMMENT '报告期',
    report_type                 VARCHAR(5)   NOT NULL  COMMENT '报告类型',
    comp_type                   VARCHAR(5)             COMMENT '公司类型',
    total_assets                FLOAT                  COMMENT '资产总计(元)',
    total_liab                  FLOAT                  COMMENT '负债合计(元)',
    total_hldr_eqy_exc_min_int  FLOAT                  COMMENT '股东权益(不含少数)(元)',
    total_hldr_eqy_inc_min_int  FLOAT                  COMMENT '股东权益(含少数)(元)',
    total_cur_assets            FLOAT                  COMMENT '流动资产合计(元)',
    total_nca                   FLOAT                  COMMENT '非流动资产合计(元)',
    total_cur_liab              FLOAT                  COMMENT '流动负债合计(元)',
    total_ncl                   FLOAT                  COMMENT '非流动负债合计(元)',
    accounts_receiv             FLOAT                  COMMENT '应收账款(元)',
    inventories                 FLOAT                  COMMENT '存货(元)',
    money_cap                   FLOAT                  COMMENT '货币资金(元)',
    PRIMARY KEY (ts_code, end_date, report_type),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='资产负债表';

-- 25. 现金流量表
CREATE TABLE IF NOT EXISTS cashflow (
    ts_code               VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    ann_date              VARCHAR(8)             COMMENT '公告日期',
    end_date              VARCHAR(8)   NOT NULL  COMMENT '报告期',
    report_type           VARCHAR(5)   NOT NULL  COMMENT '报告类型',
    comp_type             VARCHAR(5)             COMMENT '公司类型',
    net_profit            FLOAT                  COMMENT '净利润(元)',
    n_cashflow_act        FLOAT                  COMMENT '经营活动现金流量净额(元)',
    n_cashflow_inv_act    FLOAT                  COMMENT '投资活动现金流量净额(元)',
    n_cash_flows_fnc_act  FLOAT                  COMMENT '筹资活动现金流量净额(元)',
    c_cash_equ_end_period FLOAT                  COMMENT '期末现金及等价物(元)',
    c_cash_equ_beg_period FLOAT                  COMMENT '期初现金及等价物(元)',
    PRIMARY KEY (ts_code, end_date, report_type),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='现金流量表';

-- 26. 财务指标
CREATE TABLE IF NOT EXISTS fina_indicator (
    ts_code          VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    ann_date         VARCHAR(8)             COMMENT '公告日期',
    end_date         VARCHAR(8)   NOT NULL  COMMENT '报告期',
    eps              FLOAT                  COMMENT '基本每股收益(元)',
    dt_eps           FLOAT                  COMMENT '稀释每股收益(元)',
    bps              FLOAT                  COMMENT '每股净资产(元)',
    roe              FLOAT                  COMMENT '净资产收益率(%)',
    roe_dt           FLOAT                  COMMENT '净资产收益率(扣非)(%)',
    roa              FLOAT                  COMMENT '总资产净利率(%)',
    current_ratio    FLOAT                  COMMENT '流动比率',
    quick_ratio      FLOAT                  COMMENT '速动比率',
    gross_margin     FLOAT                  COMMENT '销售毛利率(%)',
    netprofit_margin FLOAT                  COMMENT '销售净利率(%)',
    debt_to_assets   FLOAT                  COMMENT '资产负债率(%)',
    op_yoy           FLOAT                  COMMENT '营业利润同比(%)',
    profit_yoy       FLOAT                  COMMENT '净利润同比(%)',
    PRIMARY KEY (ts_code, end_date),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='财务指标';

-- =====================================================
-- 高级数据表 (5000+积分)
-- =====================================================

-- 27. 沪深股通持股明细
CREATE TABLE IF NOT EXISTS hk_hold (
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    ts_code     VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    name        VARCHAR(50)            COMMENT '股票名称',
    vol         FLOAT                  COMMENT '持股数量(股)',
    ratio       FLOAT                  COMMENT '持股占比(%)',
    exchange    VARCHAR(5)             COMMENT '类型(SH/SZ)',
    PRIMARY KEY (trade_date, ts_code),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='沪深股通持股明细';

-- 28. 每日筹码及胜率
CREATE TABLE IF NOT EXISTS cyq_perf (
    ts_code      VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    trade_date   VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    his_low      FLOAT                  COMMENT '历史最低价',
    his_high     FLOAT                  COMMENT '历史最高价',
    cost_5pct    FLOAT                  COMMENT '5%成本价',
    cost_15pct   FLOAT                  COMMENT '15%成本价',
    cost_50pct   FLOAT                  COMMENT '50%成本价',
    cost_85pct   FLOAT                  COMMENT '85%成本价',
    cost_95pct   FLOAT                  COMMENT '95%成本价',
    weight_avg   FLOAT                  COMMENT '加权平均成本',
    winner_rate  FLOAT                  COMMENT '胜率(%)',
    PRIMARY KEY (ts_code, trade_date),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='每日筹码及胜率';

-- 29. 股票技术面因子
CREATE TABLE IF NOT EXISTS stk_factor (
    ts_code     VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    trade_date  VARCHAR(8)   NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    close       FLOAT                  COMMENT '收盘价',
    open        FLOAT                  COMMENT '开盘价',
    high        FLOAT                  COMMENT '最高价',
    low         FLOAT                  COMMENT '最低价',
    vol         FLOAT                  COMMENT '成交量(手)',
    amount      FLOAT                  COMMENT '成交额(千元)',
    macd_dif    FLOAT                  COMMENT 'MACD_DIF',
    macd_dea    FLOAT                  COMMENT 'MACD_DEA',
    macd        FLOAT                  COMMENT 'MACD',
    kdj_k       FLOAT                  COMMENT 'KDJ_K',
    kdj_d       FLOAT                  COMMENT 'KDJ_D',
    kdj_j       FLOAT                  COMMENT 'KDJ_J',
    rsi_6       FLOAT                  COMMENT 'RSI_6',
    rsi_12      FLOAT                  COMMENT 'RSI_12',
    rsi_24      FLOAT                  COMMENT 'RSI_24',
    boll_upper  FLOAT                  COMMENT 'BOLL上轨',
    boll_mid    FLOAT                  COMMENT 'BOLL中轨',
    boll_lower  FLOAT                  COMMENT 'BOLL下轨',
    cci         FLOAT                  COMMENT 'CCI指标',
    PRIMARY KEY (ts_code, trade_date),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='股票技术面因子';

-- 30. 大宗交易
CREATE TABLE IF NOT EXISTS block_trade (
    ts_code     VARCHAR(10)   NOT NULL  COMMENT '股票代码',
    trade_date  VARCHAR(8)    NOT NULL  COMMENT '交易日期(YYYYMMDD)',
    name        VARCHAR(50)             COMMENT '股票名称',
    price       FLOAT                   COMMENT '成交价',
    vol         FLOAT                   COMMENT '成交量(万股)',
    amount      FLOAT                   COMMENT '成交金额(万元)',
    buyer       VARCHAR(100)            COMMENT '买方营业部',
    seller      VARCHAR(100)            COMMENT '卖方营业部',
    PRIMARY KEY (trade_date, ts_code, buyer(50), seller(50)),
    INDEX idx_trade_date (trade_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='大宗交易';

-- 31. 股东人数
CREATE TABLE IF NOT EXISTS stk_holdernumber (
    ts_code           VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    ann_date          VARCHAR(8)             COMMENT '公告日期',
    end_date          VARCHAR(8)   NOT NULL  COMMENT '报告期',
    holder_num        INT                    COMMENT '股东总数',
    holder_num_change INT                    COMMENT '股东人数变化',
    holder_num_ratio  FLOAT                  COMMENT '股东人数变化比例(%)',
    holder_num_pct    FLOAT                  COMMENT '较上期变动幅度(%)',
    PRIMARY KEY (ts_code, end_date),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='股东人数';

-- 32. 前十大股东
CREATE TABLE IF NOT EXISTS top10_holders (
    ts_code      VARCHAR(10)   NOT NULL  COMMENT '股票代码',
    ann_date     VARCHAR(8)              COMMENT '公告日期',
    end_date     VARCHAR(8)    NOT NULL  COMMENT '报告期',
    holder_name  VARCHAR(200)  NOT NULL  COMMENT '股东名称',
    hold_amount  FLOAT                   COMMENT '持股数量(股)',
    hold_ratio   FLOAT                   COMMENT '持股比例(%)',
    hold_change  FLOAT                   COMMENT '持股变化(股)',
    holder_type  VARCHAR(20)             COMMENT '股东类型',
    PRIMARY KEY (ts_code, end_date, holder_name(100)),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='前十大股东';

-- 33. 前十大流通股东
CREATE TABLE IF NOT EXISTS top10_floatholders (
    ts_code      VARCHAR(10)   NOT NULL  COMMENT '股票代码',
    ann_date     VARCHAR(8)              COMMENT '公告日期',
    end_date     VARCHAR(8)    NOT NULL  COMMENT '报告期',
    holder_name  VARCHAR(200)  NOT NULL  COMMENT '股东名称',
    hold_amount  FLOAT                   COMMENT '持股数量(股)',
    hold_ratio   FLOAT                   COMMENT '持股比例(%)',
    hold_change  FLOAT                   COMMENT '持股变化(股)',
    holder_type  VARCHAR(20)             COMMENT '股东类型',
    PRIMARY KEY (ts_code, end_date, holder_name(100)),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='前十大流通股东';

-- 34. 分红送股
CREATE TABLE IF NOT EXISTS dividend (
    ts_code       VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    ann_date      VARCHAR(8)             COMMENT '公告日期',
    end_date      VARCHAR(8)   NOT NULL  COMMENT '分红年度',
    div_proc      VARCHAR(20)            COMMENT '实施进度',
    stk_div       FLOAT                  COMMENT '每股送股比例',
    stk_bo_rate   FLOAT                  COMMENT '每股转增比例',
    stk_co_rate   FLOAT                  COMMENT '每股配股比例',
    cash_div      FLOAT                  COMMENT '每股分红(税后)',
    cash_div_tax  FLOAT                  COMMENT '每股分红(税前)',
    record_date   VARCHAR(8)             COMMENT '股权登记日',
    ex_date       VARCHAR(8)             COMMENT '除权除息日',
    pay_date      VARCHAR(8)             COMMENT '派息日',
    PRIMARY KEY (ts_code, end_date),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='分红送股';

-- 35. 限售股解禁
CREATE TABLE IF NOT EXISTS share_float (
    ts_code      VARCHAR(10)   NOT NULL  COMMENT '股票代码',
    ann_date     VARCHAR(8)              COMMENT '公告日期',
    float_date   VARCHAR(8)    NOT NULL  COMMENT '解禁日期',
    float_share  FLOAT                   COMMENT '解禁数量(万股)',
    float_ratio  FLOAT                   COMMENT '解禁比例(%)',
    holder_name  VARCHAR(200)            COMMENT '股东名称',
    share_type   VARCHAR(50)             COMMENT '股份类型',
    PRIMARY KEY (ts_code, float_date, holder_name(100)),
    INDEX idx_float_date (float_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='限售股解禁';

-- 36. 股权质押统计
CREATE TABLE IF NOT EXISTS pledge_stat (
    ts_code       VARCHAR(10)  NOT NULL  COMMENT '股票代码',
    end_date      VARCHAR(8)   NOT NULL  COMMENT '截止日期',
    pledge_count  INT                    COMMENT '质押次数',
    unrest_pledge FLOAT                  COMMENT '无限售股质押数量(万股)',
    rest_pledge   FLOAT                  COMMENT '限售股份质押数量(万股)',
    total_share   FLOAT                  COMMENT '总股本(万股)',
    pledge_ratio  FLOAT                  COMMENT '质押比例(%)',
    PRIMARY KEY (ts_code, end_date),
    INDEX idx_end_date (end_date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='股权质押统计';
"""

print("📋 建表SQL脚本已准备好")
print("执行以下命令创建所有数据表:")

📋 建表SQL脚本已准备好
执行以下命令创建所有数据表:


In [54]:
# 执行建表脚本
import re

def execute_create_tables(sql_script):
    """解析并执行建表SQL脚本"""
    # 移除SQL注释 (-- 开头的行)
    lines = sql_script.split('\n')
    clean_lines = [line for line in lines if not line.strip().startswith('--')]
    clean_sql = '\n'.join(clean_lines)
    
    # 按分号分割语句
    statements = clean_sql.split(';')
    
    created_tables = []
    with engine.connect() as conn:
        for statement in statements:
            statement = statement.strip()
            if statement and 'CREATE TABLE' in statement:
                try:
                    # 提取表名
                    match = re.search(r'CREATE TABLE IF NOT EXISTS\s+(\w+)', statement)
                    if match:
                        table_name = match.group(1)
                    else:
                        table_name = "unknown"
                    
                    conn.execute(text(statement))
                    created_tables.append(table_name)
                    print(f"  ✅ 创建表 {table_name}")
                except Exception as e:
                    print(f"  ❌ 执行失败: {e}")
        conn.commit()
    
    print(f"\n✅ 共创建 {len(created_tables)} 张表!")
    return created_tables

# 执行建表
# 首次使用时取消下面的注释并运行
# created = execute_create_tables(CREATE_TABLES_SQL)

print("⚠️ 请取消上方注释 `created = execute_create_tables(CREATE_TABLES_SQL)` 并运行以创建数据表")

⚠️ 请取消上方注释 `created = execute_create_tables(CREATE_TABLES_SQL)` 并运行以创建数据表


## 数据同步说明

按 DataFetch 模块组织，详细列出每个数据表的字段和同步信息。

---

### 1. FetchBasic - 基础数据 (120-2000积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| stock_basic | StockBasicFetch | pro.stock_basic() | 2000 | 每周 |
| trade_cal | TradeCalFetch | pro.trade_cal() | 2000 | 每年 |
| stock_company | StockCompanyFetch | pro.stock_company() | 120 | 每周 |

**stock_basic 字段**: `ts_code`(股票代码), `symbol`(代码), `name`(名称), `area`(地域), `industry`(行业), `market`(市场), `exchange`(交易所), `is_hs`(沪深港通), `list_date`(上市日期)

**trade_cal 字段**: `exchange`(交易所), `cal_date`(日期), `is_open`(是否交易)

**stock_company 字段**: `ts_code`, `com_name`(公司名称), `reg_capital`(注册资本), `province`, `city`, `employees`(员工数), `main_business`(主营业务), `business_scope`(经营范围)

---

### 2. FetchDaily - A股日线行情 (120-2000积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| daily_kline | StockDailyFetch | pro.daily() | 120 | 每日16:00 |
| daily_basic | StockDailyBasicFetch | pro.daily_basic() | 2000 | 每日16:30 |

**daily_kline 字段**: `ts_code`(股票代码), `trade_date`(交易日期), `open`(开盘价), `high`(最高价), `low`(最低价), `close`(收盘价), `pre_close`(昨收价), `change`(涨跌额), `pct_chg`(涨跌幅%), `vol`(成交量/手), `amount`(成交额/千元)

**daily_basic 字段**: `ts_code`, `trade_date`, `close`, `turnover_rate`(换手率), `turnover_rate_f`(自由流通换手率), `volume_ratio`(量比), `pe`(市盈率), `pe_ttm`, `pb`(市净率), `ps`(市销率), `ps_ttm`, `dv_ratio`(股息率), `dv_ttm`, `total_share`(总股本/万), `float_share`(流通股本), `free_share`(自由流通股本), `total_mv`(总市值/万), `circ_mv`(流通市值)

---

### 3. FetchMoneyFlow - 资金流向 (120-2000积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| moneyflow | MoneyFlowFetch | pro.moneyflow() | 2000 | 每日16:30 |
| moneyflow_hsgt | MoneyFlowHSGTFetch | pro.moneyflow_hsgt() | 120 | 每日17:00 |
| hsgt_top10 | HSGTTop10Fetch | pro.hsgt_top10() | 120 | 每日17:00 |

**moneyflow 字段**: `ts_code`, `trade_date`, `buy_sm_vol/amount`(小单买入), `sell_sm_vol/amount`(小单卖出), `buy_md_vol/amount`(中单买入), `sell_md_vol/amount`(中单卖出), `buy_lg_vol/amount`(大单买入), `sell_lg_vol/amount`(大单卖出), `buy_elg_vol/amount`(特大单买入), `sell_elg_vol/amount`(特大单卖出), `net_mf_vol/amount`(净流入)

**moneyflow_hsgt 字段**: `trade_date`, `ggt_ss`(港股通沪), `ggt_sz`(港股通深), `hgt`(沪股通), `sgt`(深股通), `north_money`(北向净流入), `south_money`(南向净流入)

**hsgt_top10 字段**: `trade_date`, `ts_code`, `name`, `close`, `change`, `rank`, `market_type`(1沪/2深/3港沪/4港深), `amount`(成交额), `net_amount`(净买入), `buy`, `sell`

---

### 4. FetchTopList - 龙虎榜 (300积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| top_list | TopListFetch | pro.top_list() | 300 | 每日18:00 |
| top_inst | TopInstFetch | pro.top_inst() | 300 | 每日18:00 |

**top_list 字段**: `trade_date`, `ts_code`, `name`, `close`, `pct_change`(涨跌幅), `turnover_rate`(换手率), `amount`(成交额), `l_sell`(龙虎榜卖出额), `l_buy`(龙虎榜买入额), `l_amount`(龙虎榜成交额), `net_amount`(净买入额), `net_rate`(净买入占比), `amount_rate`(成交额占比), `float_values`(流通市值), `reason`(上榜原因)

**top_inst 字段**: `trade_date`, `ts_code`, `exalter`(营业部名称), `buy`(买入额/万), `buy_rate`(买入占比), `sell`(卖出额/万), `sell_rate`(卖出占比), `net_buy`(净买入), `side`(买卖方向BUY/SELL)

---

### 5. FetchLimit - 涨跌停 (120-5000积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| limit_list | LimitListFetch | pro.limit_list_d() | 5000 | 每日16:30 |
| stk_limit | StkLimitFetch | pro.stk_limit() | 120 | 每日16:30 |

**limit_list 字段**: `trade_date`, `ts_code`, `name`, `close`, `pct_chg`(涨跌幅), `amp`(振幅), `fc_ratio`(封单金额/流通市值), `fl_ratio`(封单手数/流通股本), `fd_amount`(封单金额/万), `first_time`(首次涨跌停时间), `last_time`(最后涨跌停时间), `open_times`(打开次数), `strth`(涨跌停强度), `limit`(U涨停/D跌停/Z炸板)

**stk_limit 字段**: `trade_date`, `ts_code`, `pre_close`(昨收价), `up_limit`(涨停价), `down_limit`(跌停价)

---

### 6. FetchIndex - 指数数据 (120-2000积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| index_daily | IndexDailyFetch | pro.index_daily() | 120 | 每日16:00 |
| index_basic | IndexBasicFetch | pro.index_basic() | 120 | 每月 |
| index_sw_daily | SWIndexDailyFetch | pro.sw_daily() | 2000 | 每日16:30 |
| index_classify | IndexClassifyFetch | pro.index_classify() | 120 | 每季度 |
| index_member | IndexMemberFetch | pro.index_member() | 120 | 每季度 |

**index_daily 字段**: `ts_code`(指数代码), `trade_date`, `close`, `open`, `high`, `low`, `pre_close`, `change`, `pct_chg`, `vol`, `amount`

**index_basic 字段**: `ts_code`, `name`, `fullname`, `market`, `publisher`(发布方), `index_type`, `category`, `base_date`(基期), `base_point`(基点), `list_date`, `weight_rule`(加权方式), `desc`, `exp_date`

**index_sw_daily 字段**: `ts_code`, `trade_date`, `name`(行业名称), `open`, `low`, `high`, `close`, `change`, `pct_change`, `vol`, `amount`, `pe`(市盈率), `pb`(市净率)

**index_classify 字段**: `index_code`, `industry_name`(行业名称), `level`(L1/L2/L3), `industry_code`, `is_pub`(是否发布指数), `parent_code`(父级代码)

**index_member 字段**: `index_code`, `index_name`, `con_code`(成分股代码), `con_name`(成分股名称), `in_date`(纳入日期), `out_date`(剔除日期), `is_new`

---

### 7. FetchHK - 港股数据 (2000积分 / 单独权限)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| hk_basic | HKBasicFetch | pro.hk_basic() | 2000 | 每周 |
| ggt_daily | GGTDailyFetch | pro.ggt_daily() | 2000 | 每日17:00 |
| hk_daily | - | pro.hk_daily() | **单独权限** | ⚠️需购买(1000元/年) |

**hk_basic 字段**: `ts_code`, `name`, `fullname`, `enname`, `cn_spell`, `market`, `list_status`(L上市/D退市/P暂停), `list_date`, `delist_date`, `trade_unit`(交易单位), `isin`, `curr_type`(货币)

**ggt_daily 字段**: `trade_date`, `buy_amount`(买入成交金额/亿元), `buy_volume`(买入成交笔数/万笔), `sell_amount`(卖出成交金额/亿元), `sell_volume`(卖出成交笔数/万笔) - 注意:市场汇总数据,非单股

---

### 8. FetchMargin - 融资融券 (120积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| margin | MarginFetch | pro.margin() | 120 | 每日18:30 |
| margin_detail | MarginDetailFetch | pro.margin_detail() | 120 | 每日18:30 |

**margin 字段**: `trade_date`, `exchange_id`(SSE/SZSE), `rzye`(融资余额), `rzmre`(融资买入额), `rzche`(融资偿还额), `rqye`(融券余额), `rqmcl`(融券卖出量), `rzrqye`(融资融券余额), `rqyl`(融券余量)

**margin_detail 字段**: `trade_date`, `ts_code`, `name`, `rzye`, `rqye`, `rzmre`, `rqyl`, `rzche`, `rqchl`(融券偿还量), `rqmcl`, `rzrqye`

---

### 9. FetchFinancial - 财务数据 (2000-5000积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| income | IncomeVipFetch | pro.income_vip() | 5000 | 财报季后 |
| balancesheet | BalanceSheetVipFetch | pro.balancesheet_vip() | 5000 | 财报季后 |
| cashflow | CashFlowVipFetch | pro.cashflow_vip() | 5000 | 财报季后 |
| fina_indicator | FinaIndicatorVipFetch | pro.fina_indicator_vip() | 5000 | 财报季后 |

**income 字段**: `ts_code`, `ann_date`(公告日期), `end_date`(报告期), `report_type`, `comp_type`, `basic_eps`(基本每股收益), `diluted_eps`(稀释每股收益), `total_revenue`(营业总收入), `revenue`(营业收入), `total_cogs`(营业总成本), `operate_profit`(营业利润), `total_profit`(利润总额), `n_income`(净利润), `n_income_attr_p`(归母净利润)

**balancesheet 字段**: `ts_code`, `ann_date`, `end_date`, `report_type`, `comp_type`, `total_assets`(总资产), `total_liab`(总负债), `total_hldr_eqy_exc_min_int`(股东权益不含少数), `total_hldr_eqy_inc_min_int`(股东权益含少数), `total_cur_assets`(流动资产), `total_nca`(非流动资产), `total_cur_liab`(流动负债), `total_ncl`(非流动负债), `accounts_receiv`(应收账款), `inventories`(存货), `money_cap`(货币资金)

**cashflow 字段**: `ts_code`, `ann_date`, `end_date`, `report_type`, `comp_type`, `net_profit`(净利润), `n_cashflow_act`(经营活动现金流), `n_cashflow_inv_act`(投资活动现金流), `n_cash_flows_fnc_act`(筹资活动现金流), `c_cash_equ_end_period`(期末现金), `c_cash_equ_beg_period`(期初现金)

**fina_indicator 字段**: `ts_code`, `ann_date`, `end_date`, `eps`(基本每股收益), `dt_eps`(稀释每股收益), `bps`(每股净资产), `roe`(净资产收益率), `roe_dt`(净资产收益率扣非), `roa`(总资产净利率), `current_ratio`(流动比率), `quick_ratio`(速动比率), `gross_margin`(毛利率), `netprofit_margin`(净利率), `debt_to_assets`(资产负债率), `op_yoy`(营业利润同比), `profit_yoy`(净利润同比)

---

### 10. FetchAdvanced - 高级数据 (2000-5000积分)

| 表名 | DataFetch类 | TuShare接口 | 积分 | 更新频率 |
|------|-------------|-------------|------|----------|
| hk_hold | HKHoldFetch | pro.hk_hold() | 5000 | 每日17:00 |
| cyq_perf | CyqPerfFetch | pro.cyq_perf() | 5000 | 每日17:00 |
| stk_factor | StkFactorFetch | pro.stk_factor_pro() | 5000 | 每日17:00 |
| block_trade | BlockTradeFetch | pro.block_trade() | 2000 | 每日18:00 |
| stk_holdernumber | StkHolderNumberFetch | pro.stk_holdernumber() | 2000 | 财报季后 |
| top10_holders | Top10HoldersFetch | pro.top10_holders() | 2000 | 财报季后 |
| top10_floatholders | Top10FloatHoldersFetch | pro.top10_floatholders() | 2000 | 财报季后 |
| dividend | DividendFetch | pro.dividend() | 2000 | 每年 |
| share_float | ShareFloatFetch | pro.share_float() | 3000 | 每周 |
| pledge_stat | PledgeStatFetch | pro.pledge_stat() | 2000 | 每周 |

**hk_hold 字段**: `trade_date`, `ts_code`, `name`, `vol`(持股数量/股), `ratio`(持股占比%), `exchange`(SH/SZ)

**cyq_perf 字段**: `ts_code`, `trade_date`, `his_low`(历史最低), `his_high`(历史最高), `cost_5pct`(5%成本价), `cost_15pct`, `cost_50pct`, `cost_85pct`, `cost_95pct`, `weight_avg`(加权平均成本), `winner_rate`(胜率%)

**stk_factor 字段**: `ts_code`, `trade_date`, `close`, `open`, `high`, `low`, `vol`, `amount`, `macd_dif`, `macd_dea`, `macd`, `kdj_k`, `kdj_d`, `kdj_j`, `rsi_6`, `rsi_12`, `rsi_24`, `boll_upper`, `boll_mid`, `boll_lower`, `cci`

**block_trade 字段**: `ts_code`, `trade_date`, `name`, `price`(成交价), `vol`(成交量/万股), `amount`(成交金额/万元), `buyer`(买方营业部), `seller`(卖方营业部)

**stk_holdernumber 字段**: `ts_code`, `ann_date`, `end_date`, `holder_num`(股东总数), `holder_num_change`(变化), `holder_num_ratio`(变化比例%), `holder_num_pct`(较上期变动幅度%)

**top10_holders/top10_floatholders 字段**: `ts_code`, `ann_date`, `end_date`, `holder_name`(股东名称), `hold_amount`(持股数量/股), `hold_ratio`(持股比例%), `hold_change`(持股变化/股), `holder_type`(股东类型)

**dividend 字段**: `ts_code`, `ann_date`, `end_date`, `div_proc`(实施进度), `stk_div`(每股送股比例), `stk_bo_rate`(每股转增比例), `stk_co_rate`(每股配股比例), `cash_div`(每股分红税后), `cash_div_tax`(税前), `record_date`(股权登记日), `ex_date`(除权除息日), `pay_date`(派息日)

**share_float 字段**: `ts_code`, `ann_date`, `float_date`(解禁日期), `float_share`(解禁数量/万股), `float_ratio`(解禁比例%), `holder_name`, `share_type`(股份类型)

**pledge_stat 字段**: `ts_code`, `end_date`, `pledge_count`(质押次数), `unrest_pledge`(无限售股质押数量/万股), `rest_pledge`(限售股份质押数量/万股), `total_share`(总股本/万股), `pledge_ratio`(质押比例%)

---

### TuShare 积分要求汇总

| 积分等级 | 可用接口 |
|---------|---------|
| 120+ | daily, stock_company, index_daily, index_basic, moneyflow_hsgt, hsgt_top10, margin, margin_detail, stk_limit, index_classify, index_member |
| 300+ | top_list, top_inst |
| 2000+ | stock_basic, trade_cal, daily_basic, moneyflow, sw_daily, hk_basic, ggt_daily, block_trade, stk_holdernumber, top10_holders, top10_floatholders, dividend, pledge_stat |
| 3000+ | share_float |
| 5000+ | limit_list_d, hk_hold, cyq_perf, stk_factor_pro, income_vip, balancesheet_vip, cashflow_vip, fina_indicator_vip |
| **单独权限** | hk_daily (港股日线) - 需购买权限(1000元/年)，非积分制 |

### 财报披露时间参考
- **一季报**: 4月1日 ~ 4月30日
- **半年报**: 7月1日 ~ 8月31日
- **三季报**: 10月1日 ~ 10月31日
- **年报**: 次年1月1日 ~ 4月30日